# 02 - Reconcile cells and nuclei

In the paper this pairing is a demonstration: on the mouse salivary gland it was used to make each nucleus fit inside one cell and to count how many cells contain one. No conclusions in the paper rest on it, and its accuracy has not been evaluated against ground truth; use it to inspect and curate, and judge the pairing on your own data. This notebook does not apply the paper's rule; it measures the overlap with its own thresholds (0.15 and 0.20) to show you where the problems are, so its percentages will not match the paper's. Notebook 03 applies the paper's rule.


You have a **cell** segmentation and a **nuclei** segmentation of the same tissue, both produced by notebook 01. In a perfect world every cell contains exactly one nucleus and every nucleus sits in exactly one cell. In practice some cells hold two nuclei (two cells were glued together, or a cell is dividing), some cells have none (a fragment, or a missed nucleus), and some nuclei straddle two cells or sit in none.

This notebook measures that, using only the two label volumes. Nothing is edited and nothing needs a ground truth. It:

1. builds the overlap between every cell and every nucleus (a sparse table, never a full matrix),
2. sorts every cell and nucleus into a type: clean 1:1 pair, cell with two or more nuclei, cell with no nucleus, orphan nucleus, nucleus shared by two cells,
3. shows how clean the pairing is and where the problems are, with plots and a slice picture coloured by type.

Notebook 03 then uses the same two volumes to make them consistent.

**Words used here.** A nucleus that *straddles* two cells lies across their boundary (the engine calls this "strung"). A cell left with no nucleus because a neighbour claimed it is a *deprived partner* (notebook 03 calls its status `no_nucleus_strung_victim`). A *merge* is a cell that holds two or more nuclei.

**What you need: two run folders from notebook 01.** A run folder is the folder that notebook 01 prints as `Output:`, that is `WORK_DIR/OUTPUT_NAME` (not `WORK_DIR` itself). You need one for the cells and one for the nuclei, both made from the same stack, so that they have the same shape.

**Making the nuclei run.** Run notebook 01 a second time:

- use the same `INPUT_TIF` (the multichannel stack),
- use a model pair trained on nuclei, and set `CHANNELS` to the nuclear stain,
- give it a different `OUTPUT_NAME`,
- leave post-processing on (`RUN_POSTPROCESS = True`, `PP_DRY_RUN = False`), so that the `*_clean` files this notebook reads exist.

## 1. Paths

Edit the two run folders (the `Output:` folders of notebook 01, as explained above). Each must contain `stage4_full_stitch/global_labels_clean.npy` and `stage4_full_stitch/label_provenance_clean.csv`, which notebook 01 writes when its clean-up step has run. If a file is reported missing, the cell lists what the folder does contain, so you can see whether you are one level too high or too low.

In [ ]:
import sys
from pathlib import Path

# Make `packages.epicycle_scaledseg` importable when the notebook is opened from the
# repository root or from tutorials/segmentation/ (not needed if you ran `pip install -e .`).
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / "packages" / "epicycle_scaledseg").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    print("Could not find the EpiCycle repository above", Path.cwd(),
          "- open the notebook from inside the repository, or run: pip install -e .")

In [ ]:
from pathlib import Path

# The two run folders from notebook 01 (each holds stage4_full_stitch/global_labels_clean.npy)
CELL_RUN_DIR   = Path(r"<EDIT>/work/cells_output")     # the CELL segmentation run
NUCLEI_RUN_DIR = Path(r"<EDIT>/work/nuclei_output")   # the NUCLEI segmentation run

CELL_LABELS_NPY   = CELL_RUN_DIR   / "stage4_full_stitch" / "global_labels_clean.npy"
NUCLEI_LABELS_NPY = NUCLEI_RUN_DIR / "stage4_full_stitch" / "global_labels_clean.npy"
CELL_PROV   = CELL_RUN_DIR   / "stage4_full_stitch" / "label_provenance_clean.csv"
NUCLEI_PROV = NUCLEI_RUN_DIR / "stage4_full_stitch" / "label_provenance_clean.csv"
for p in (CELL_LABELS_NPY, NUCLEI_LABELS_NPY, CELL_PROV, NUCLEI_PROV):
    print(p, "OK" if p.exists() else "** MISSING **")
    if not p.exists():
        folder = p.parent if p.parent.exists() else (CELL_RUN_DIR if p in (CELL_LABELS_NPY, CELL_PROV) else NUCLEI_RUN_DIR)
        found = sorted(q.name for q in folder.iterdir()) if folder.exists() else "(the folder does not exist)"
        print("   that folder contains:", found)

## 2. Load and check that the two volumes line up

The volumes are opened memory-mapped, so the whole volume is never read into RAM at once: later cells read them a few planes at a time. Their shapes must match. If they do not, the two runs are not in the same coordinate frame and an overlap would mean nothing. Matching shapes are necessary but not sufficient (two runs made from the same image always have the same shape), so section 7 checks the alignment properly.

In [ ]:
import numpy as np

assert "<EDIT>" not in str(CELL_RUN_DIR) and "<EDIT>" not in str(NUCLEI_RUN_DIR), "edit the run folders in section 1"
cells_vol  = np.load(str(CELL_LABELS_NPY),   mmap_mode="r")   # never fully loaded
nuclei_vol = np.load(str(NUCLEI_LABELS_NPY), mmap_mode="r")
print("cell   vol:", cells_vol.shape, cells_vol.dtype)
print("nuclei vol:", nuclei_vol.shape, nuclei_vol.dtype)
assert cells_vol.shape == nuclei_vol.shape, (
    "CO-REGISTRATION FAIL: shapes differ - the cell and nuclei runs must be on the "
    "same stack, tiling and Z range. Fix this before going on.")
print("shapes match (necessary, not sufficient): section 7 checks that the two volumes are actually aligned")

## 3. Build the cell-by-nucleus overlap table

One pass over all voxels: for every voxel that is in both a cell and a nucleus, count the (cell id, nucleus id) pair. The work is split into slabs of Z planes and run on several threads, then added up. The result is **sparse**: each cell overlaps only a few nuclei, so we keep an edge table (cell, nucleus, shared voxels), never a dense cells-by-nuclei array.

**Memory rule.** Each thread holds one slab at a time (about 25 bytes per voxel of the slab). The slab height is chosen from the size of one plane so that a slab stays near `SLAB_VOXELS`, and the number of threads is capped so that all slabs together stay inside `RAM_BUDGET_GB`. Raise the budget if you have the memory; the answer does not change.

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

RAM_BUDGET_GB = 8          # memory the counting threads may use together
SLAB_VOXELS = 50_000_000   # voxels per slab (about 1.2 GB per thread while counting)
Z, Y, X = cells_vol.shape
SLAB = max(1, int(SLAB_VOXELS // (Y * X)))          # Z planes counted at a time
per_thread = SLAB * Y * X * 25                       # bytes
N_WORKERS = int(max(1, min((os.cpu_count() or 2) - 1, RAM_BUDGET_GB * 1e9 // per_thread)))


def _overlap_chunk(zr):
    """Count (cell id, nucleus id) pairs over the Z range zr = (z0, z1)."""
    z0, z1 = zr
    local = {}
    for zs in range(z0, z1, SLAB):
        ze = min(zs + SLAB, z1)
        cz = np.asarray(cells_vol[zs:ze])
        nz = np.asarray(nuclei_vol[zs:ze])
        both = (cz > 0) & (nz > 0)
        if both.any():
            # one 64-bit key per voxel: cell id in the high half, nucleus id in the low half
            key = (cz[both].astype(np.int64) << 32) | nz[both].astype(np.int64)
            uniq, cnt = np.unique(key, return_counts=True)
            for kk, c in zip(uniq.tolist(), cnt.tolist()):
                k = (kk >> 32, kk & 0xFFFFFFFF)
                local[k] = local.get(k, 0) + int(c)
    return local


# slabs of Z planes, one per task (a few more tasks than threads, to balance the load)
nch = max(1, min(Z, N_WORKERS * 4))
b = np.linspace(0, Z, nch + 1).astype(int)
chunks = [(int(b[k]), int(b[k + 1])) for k in range(nch) if b[k + 1] > b[k]]

overlap = defaultdict(int)
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    for local in pool.map(_overlap_chunk, chunks):
        for k, v in local.items():
            overlap[k] += v
print(f"{len(overlap):,} cell-nucleus overlap pairs  ({len(chunks)} chunks, {N_WORKERS} threads, "
      f"{SLAB} planes per slab)")

## 4. The edge table and a first classification

For every overlapping pair we compute **containment**: the fraction of the nucleus that lies inside that cell. A nucleus "belongs" to a cell if at least `CONTAINMENT_THR` of it is inside. Counting nuclei per cell, and cells per nucleus, gives the first classification:

- **clean_1to1**: the cell has exactly one nucleus,
- **cell_no_nucleus** and **cell_multi_nucleus**: none, or two or more,
- **nucleus_ok**, **orphan_nucleus**, **nucleus_multi_cell**: the same idea from the nucleus side.

The next sections refine this, because a plain threshold cannot see every case. In particular, a nucleus that lies half in each of two cells is not counted as "in" either one at a threshold of 0.5, so `nucleus_multi_cell` stays near zero here; section 6 counts straddling nuclei properly, and from there on the counts to report are the ones from section 6.

In [ ]:
cell_prov = pd.read_csv(CELL_PROV); nuc_prov = pd.read_csv(NUCLEI_PROV)
cell_ids    = cell_prov.loc[cell_prov.voxel_count > 0, "dense_id"].astype(int).tolist()
nucleus_ids = nuc_prov.loc[nuc_prov.voxel_count > 0, "dense_id"].astype(int).tolist()
nuc_vox = dict(zip(nuc_prov.dense_id.astype(int), nuc_prov.voxel_count.astype(int)))

edges = pd.DataFrame(
    [(c, n, s, nuc_vox.get(n, 0)) for (c, n), s in overlap.items()],
    columns=["cell_id", "nucleus_id", "shared_vox", "nucleus_vox"])
edges["containment"] = edges.shared_vox / edges.nucleus_vox.clip(lower=1)

CONTAINMENT_THR = 0.5   # a nucleus 'belongs' to a cell if >= this fraction is inside it
matched = edges[edges.containment >= CONTAINMENT_THR]
nuc_per_cell = matched.groupby("cell_id").size().reindex(cell_ids, fill_value=0)
cell_per_nuc = matched.groupby("nucleus_id").size().reindex(nucleus_ids, fill_value=0)

cell_case = nuc_per_cell.map(lambda n: "clean_1to1" if n == 1 else ("cell_no_nucleus" if n == 0 else "cell_multi_nucleus"))
nuc_case  = cell_per_nuc.map(lambda n: "nucleus_ok" if n == 1 else ("orphan_nucleus" if n == 0 else "nucleus_multi_cell"))

nC, nN = len(cell_ids), len(nucleus_ids)
print("=== CELL side (rows) ===")
for k, v in cell_case.value_counts().items():
    print(f"  {k:20s} {v:8,d}  ({100 * v / nC:5.1f}%)")
print("=== NUCLEUS side (cols) ===")
for k, v in nuc_case.value_counts().items():
    print(f"  {k:20s} {v:8,d}  ({100 * v / nN:5.1f}%)")
print(f"\nShare of cells with exactly one nucleus (clean 1:1) at this threshold: {100 * (cell_case == 'clean_1to1').mean():.1f}% of cells are clean 1:1")

## 5. How much of "no nucleus" is just the threshold?

The cutoff in section 4 rejects genuine partial matches, which inflates "cell with no nucleus". Here we plot, for each cell, the containment of its **best** nucleus, and re-count the case mix at several thresholds. Where the clean share levels off is a sensible threshold. Whatever "no nucleus" remains at a low threshold is the genuinely unmatched group (fragments, or a nucleus that was missed).

In [ ]:
# Threshold diagnostic: (a) the distribution of each cell's BEST containment, and (b) the case
# mix at several thresholds, so a threshold artefact can be told apart from a genuine anomaly.
# On well-registered data most cells have a best containment close to 1.
import matplotlib.pyplot as plt

# best (max) containment per cell over its overlapping nuclei
best_cont = edges.groupby('cell_id').containment.max().reindex(cell_ids, fill_value=0.0)
print(f"per-cell BEST containment: median={best_cont.median():.2f}  "
      f"mean={best_cont.mean():.2f}  (frac==0, truly no overlap: {(best_cont==0).mean():.1%})")

plt.figure(figsize=(7, 3.2))
plt.hist(best_cont.values, bins=np.linspace(0, 1, 51))
for thr in (0.2, 0.3, 0.5):
    plt.axvline(thr, ls='--', lw=1, c='r' if thr == 0.5 else 'k')
plt.title('per-cell best nucleus containment (red dashed = current 0.5 cutoff)')
plt.xlabel('best containment'); plt.ylabel('# cells'); plt.show()

print('\nCASE MIX vs containment threshold:')
print(f"{'thr':>5} {'clean_1to1':>11} {'no_nucleus':>11} {'multi_nucleus':>14} {'orphan_nuc':>11}")
for thr in (0.1, 0.2, 0.3, 0.4, 0.5):
    m = edges[edges.containment >= thr]
    npc = m.groupby('cell_id').size().reindex(cell_ids, fill_value=0)
    cpn = m.groupby('nucleus_id').size().reindex(nucleus_ids, fill_value=0)
    clean = int((npc == 1).sum()); nonuc = int((npc == 0).sum()); multi = int((npc >= 2).sum())
    orph = int((cpn == 0).sum())
    print(f"{thr:>5.1f} {clean:>11,} {nonuc:>11,} {multi:>14,} {orph:>11,}  "
          f"({100*clean/len(cell_ids):.1f}% clean)")
print('\nPick the threshold where the clean share levels off: below it you start absorbing'
      ' incidental touches. The no_nucleus count that remains at a low threshold is the'
      ' genuine fragment / missed-nucleus group.')

## 6. A better classification: assign each nucleus to its best cell

Containment is `shared voxels / nucleus voxels`, so for one nucleus the containments across cells add up to at most 1. With a threshold of 0.5 or more, a nucleus that straddles two cells can match **at most one** of them, so the threshold method cannot see straddling nuclei at all and it also under-counts merges.

The fix: give each nucleus to the cell that holds most of it (its best cell), as long as at least `FLOOR` of it is inside (otherwise it is an orphan). Then count by that assignment. A separate **straddling test** looks at the second-largest cell share of each nucleus: a nucleus with `TAU_STRUNG` or more of its volume in a second cell sits across a boundary.

From here on, the cell types are:

- **clean**: one nucleus assigned,
- **merge**: two or more nuclei assigned (two cells glued together, or a dividing cell),
- **no nucleus**: none assigned,
- **orphan**: a nucleus with no home.

In [ ]:
# ASSIGNMENT-BASED CLASSIFICATION (replaces the threshold-only count of section 4)
# Why: containment = shared / nucleus voxels, so per nucleus the containments across cells sum to
# at most 1. At a threshold of 0.5 or more a nucleus straddling 2 cells matches at most one, so the
# threshold method cannot see straddling nuclei and under-counts merges. Instead: assign each
# nucleus to its best cell, then count by assignment, with a separate straddling test.

# -- 0. sanity checks ------------------------------------------------------------
n_gt1 = int((edges.containment > 1.001).sum())
print(f"edges with containment>1 (label ids out of step with the volumes): {n_gt1}  (want 0)")
assert (edges.nucleus_vox > 0).all(), "an overlap with a nucleus of zero voxels: the provenance table does not match the label volume"

# -- 1. assign each nucleus to its single best cell -------------------------------
FLOOR = 0.15   # below this the nucleus has no real home (a true orphan)
ranked = edges.sort_values('containment', ascending=False)
best = ranked.drop_duplicates('nucleus_id')                 # nucleus -> its top cell
assigned = best[best.containment >= FLOOR].copy()
home = dict(zip(assigned.nucleus_id, assigned.cell_id))

# -- 2. nuclei ASSIGNED per cell (merge = >=2) ----------------------------------
nuclei_assigned = assigned.groupby('cell_id').nucleus_id.nunique().reindex(cell_ids, fill_value=0)

# -- 3. straddling test: per nucleus, the SECOND-largest cell share --------------
second = (ranked.groupby('nucleus_id').containment
                .apply(lambda s: float(s.iloc[1]) if len(s) > 1 else 0.0)
                .reindex(nucleus_ids, fill_value=0.0))
TAU_STRUNG = 0.20   # a nucleus with at least 20% of its volume in a 2nd cell straddles a boundary
n_strung = int((second >= TAU_STRUNG).sum())

# -- 4. tallies ------------------------------------------------------------------
clean  = int((nuclei_assigned == 1).sum())
merges = int((nuclei_assigned >= 2).sum())
nonuc  = int((nuclei_assigned == 0).sum())
orphan = int(len(nucleus_ids) - assigned.nucleus_id.nunique())
nC, nN = len(cell_ids), len(nucleus_ids)
print(f"\n=== assignment-based classification (floor={FLOOR}) ===")
print(f"  clean 1:1            {clean:8,} ({100*clean/nC:5.1f}% of cells)")
print(f"  MERGE (>=2 nuclei)   {merges:8,} ({100*merges/nC:5.1f}%)   <- the threshold method under-counts these")
print(f"  cell, no nucleus     {nonuc:8,} ({100*nonuc/nC:5.1f}%)")
print(f"  orphan nucleus       {orphan:8,} ({100*orphan/nN:5.1f}% of nuclei)")
print(f"  STRADDLING nuclei (2nd cell share >= {TAU_STRUNG}): {n_strung:,}  ({100*n_strung/nN:.1f}% of nuclei)")
print("     ^ the threshold method cannot see these at all")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(second.values[second.values > 0.01], bins=np.linspace(0, 0.6, 40))
ax[0].axvline(TAU_STRUNG, ls='--', c='r'); ax[0].set(title='nucleus 2nd-largest cell share (straddling evidence)',
          xlabel='2nd cell-share', ylabel='# nuclei (with any 2nd touch)')
ax[1].hist(np.clip(nuclei_assigned.values, 0, 4), bins=np.arange(-0.5, 5, 1), rwidth=0.8, color='C2')
ax[1].set(title='nuclei assigned per cell (merge = 2 or more)', xlabel='# assigned nuclei', ylabel='# cells')
plt.tight_layout(); plt.show()

## 7. Are the two volumes really aligned?

Equal shapes do not prove the volumes are aligned. For every assigned nucleus and its cell, we measure the offset between the centre of the nucleus's bounding box and the centre of the cell's bounding box, in voxels. A single nucleus sits off-centre in its cell, but across the population the **median** offset should be about zero on every axis.

**Pass rule:** the median offset is under about 1 to 2 voxels on all three axes. The cell below prints PASS or CHECK. If it prints CHECK, the two runs are probably shifted against each other (the shift would otherwise be mistaken for noisy segmentation): check that both runs used the same `INPUT_TIF`, the same axes and the same Z range, and re-run notebook 01 for the run that differs.

In [ ]:
# Registration check: equal shapes do not prove alignment. For each assigned nucleus and its cell,
# the centre offset should be about 0 on the median. A systematic offset means the runs are shifted.
cpc = cell_prov.assign(z=(cell_prov.z_min+cell_prov.z_max)/2, y=(cell_prov.y_min+cell_prov.y_max)/2,
                       x=(cell_prov.x_min+cell_prov.x_max)/2).set_index('dense_id')
npc = nuc_prov.assign(z=(nuc_prov.z_min+nuc_prov.z_max)/2, y=(nuc_prov.y_min+nuc_prov.y_max)/2,
                      x=(nuc_prov.x_min+nuc_prov.x_max)/2).set_index('dense_id')
off = np.array([[npc.z[n]-cpc.z[c], npc.y[n]-cpc.y[c], npc.x[n]-cpc.x[c]]
                for n, c in home.items() if n in npc.index and c in cpc.index])
med = np.median(off, axis=0); iqr = np.percentile(off, 75, axis=0) - np.percentile(off, 25, axis=0)
print(f"bounding-box centre offset, nucleus minus its cell (voxels, ZYX): median={med.round(2)}  IQR={iqr.round(1)}")
print("  PASS" if (np.abs(med) < 2).all() else "  CHECK: a median offset of 2 voxels or more on some axis: the runs may be shifted (see the text above)")

## 8. How many "no nucleus" cells are really a shared nucleus?

Every straddling nucleus is given to one cell, so the other cell it touches is left with "no nucleus" even though its nucleus exists. This section splits the straddling count by how much of the nucleus lies in the second cell:

- a small share (0.20 to 0.30) is usually a nucleus poking over a boundary, which is harmless,
- a middle share (0.30 to 0.45) is ambiguous,
- a share of 0.45 or more means the nucleus is genuinely cut in half by a cell boundary.

It then counts how many "no nucleus" cells are such **deprived partners**. Those are not fragments or missed nuclei: their nucleus is there and a neighbour claimed it. Notebook 03 fixes them by cutting the nucleus at the boundary.

In [ ]:
# Break the straddling count down by band, and count the "no nucleus" cells that are really the
# other half of a shared nucleus. A threshold of 0.2 includes nuclei merely poking into a neighbour;
# the genuine disputes are the high second-share tail.

# -- 1. straddling nuclei by second-share band --------------------------------
sb = second[second >= TAU_STRUNG]
print('straddling nuclei by 2nd-largest cell share:')
for lo, hi, lbl in [(0.20, 0.30, '0.20-0.30  probably just touching the boundary (mostly harmless)'),
                    (0.30, 0.45, '0.30-0.45  ambiguous'),
                    (0.45, 1.01, '>=0.45     genuinely cut by the boundary -> a real conflict')]:
    n = int(((sb >= lo) & (sb < hi)).sum())
    print(f'  {lbl}: {n:,}')
genuine = int((sb >= 0.45).sum())
print(f'  -> ~{genuine:,} are clear straddlers (vs {len(sb):,} at TAU_STRUNG (0.2))')

# -- 2. straddling <-> no-nucleus link ---------------------------------------
strung_ids = set(second[second >= TAU_STRUNG].index)
nonuc_cells = set(nuclei_assigned[nuclei_assigned == 0].index)
e2 = edges[(edges.containment >= TAU_STRUNG) & (edges.nucleus_id.isin(strung_ids))].copy()
e2['home'] = e2.nucleus_id.map(home)
deprived = set(e2[(e2.cell_id != e2.home) & (e2.cell_id.isin(nonuc_cells))].cell_id)
print(f'\nno-nucleus cells = {len(nonuc_cells):,}')
print(f"  of which DEPRIVED PARTNERS of a straddling nucleus: {len(deprived):,} "
      f"({100*len(deprived)/max(len(nonuc_cells),1):.0f}%)")
print("  -> these are NOT fragments or missed nuclei: their nucleus exists but a neighbour")
print("     claimed it. Cutting the nucleus at the boundary rescues them (notebook 03).")
print(f"     The remaining {len(nonuc_cells)-len(deprived):,} are the true fragment / missed-nucleus group (see section 11).")

## 9. Nuclei per cell and cells per nucleus

The quick picture of how clean the pairing is, using the assignment of section 6. The left histogram counts the nuclei assigned to each cell; the right one counts, for each nucleus, the cells that hold at least `TAU_STRUNG` of it. Each should be one tall bar at 1.

In [ ]:
import matplotlib.pyplot as plt
cells_per_nuc = (edges[edges.containment >= TAU_STRUNG].groupby('nucleus_id').size()
                 .reindex(nucleus_ids, fill_value=0))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(np.clip(nuclei_assigned.values, 0, 5), bins=np.arange(-0.5, 6, 1), rwidth=0.8)
ax[0].set(title="nuclei per cell (clean = spike at 1)", xlabel="# nuclei", ylabel="# cells")
ax[1].hist(np.clip(cells_per_nuc.values, 0, 5), bins=np.arange(-0.5, 6, 1), rwidth=0.8, color="C1")
ax[1].set(title=f"cells holding at least {TAU_STRUNG:.0%} of a nucleus (clean = spike at 1)",
          xlabel="# cells", ylabel="# nuclei")
plt.tight_layout(); plt.show()

## 10. A block of the overlap table, ordered so clean pairs sit on the diagonal

A readable version of the cells-by-nuclei table for a small region of the tissue (the densest patch is picked automatically, so you never land in the background). Cells and nuclei are ordered by a greedy best match, so clean 1:1 pairs fall on a bright diagonal. A second bright square in a **row** is a cell with two nuclei (a merge). A second bright square in a **column** is a nucleus over two cells (straddling).

In [ ]:
# Sub-region heatmap, ordered by a greedy cell-nucleus match so 1:1 pairs sit on the diagonal.
# The region is centred automatically on the densest patch of tissue.
cc = cell_prov.assign(cz=(cell_prov.z_min + cell_prov.z_max) / 2,
                      cy=(cell_prov.y_min + cell_prov.y_max) / 2,
                      cx=(cell_prov.x_min + cell_prov.x_max) / 2).set_index("dense_id")
cc = cc.loc[cc.index.isin(cell_ids)]   # real cells only

BOX = 250            # half-extent in Y and X (voxels); shrink it for fewer cells in the block
ZHALF = 30           # half-extent in Z
# Find the densest YX location: 2D-histogram the centroids, take the peak bin.
# coarse 2D histogram of the cell centres; its peak bin is the densest patch
H2, yedges, xedges = np.histogram2d(cc.cy, cc.cx, bins=40)
iy, ix = np.unravel_index(np.argmax(H2), H2.shape)
cyc = 0.5 * (yedges[iy] + yedges[iy + 1]); cxc = 0.5 * (xedges[ix] + xedges[ix + 1])
czc = float(cc.cz.median())
REGION = dict(z=(czc - ZHALF, czc + ZHALF), y=(cyc - BOX, cyc + BOX), x=(cxc - BOX, cxc + BOX))
print(f"auto region (densest patch): z~{czc:.0f}, y~{cyc:.0f}, x~{cxc:.0f}  (+/-{BOX} YX, +/-{ZHALF} Z)")

sel = cc[(cc.cz.between(*REGION['z'])) & (cc.cy.between(*REGION['y'])) & (cc.cx.between(*REGION['x']))].index
sub = edges[edges.cell_id.isin(sel)]
print(f"{len(sel)} cells in the block, {sub.nucleus_id.nunique()} overlapping nuclei")
csel = sorted(sub.cell_id.unique()); nsel = sorted(sub.nucleus_id.unique())
ci = {c: i for i, c in enumerate(csel)}; ni = {n: i for i, n in enumerate(nsel)}
M = np.zeros((len(csel), len(nsel)))
for c, n, val in sub[['cell_id', 'nucleus_id', 'containment']].itertuples(index=False):
    M[ci[c], ni[n]] = val

# Greedy diagonalize: strongest-matching cells first; each cell's best nucleus takes
# the next column slot. Leftover (unmatched) nuclei go to the end.
row_order, col_order, used = [], [], set()
for _, i in sorted(((M[i].max() if M.shape[1] else 0.0, i) for i in range(len(csel))), reverse=True):
    row_order.append(i)
    if M.shape[1]:
        j = int(np.argmax(M[i]))
        if M[i, j] > 0 and j not in used:
            col_order.append(j); used.add(j)
col_order += [j for j in range(len(nsel)) if j not in used]
Mord = M[np.ix_(row_order, col_order)] if (row_order and col_order) else M

import matplotlib.pyplot as plt
if Mord.size:
    plt.figure(figsize=(7, 7)); plt.imshow(Mord, aspect="auto", cmap="magma")
    plt.xlabel("nuclei (co-ordered to the match)"); plt.ylabel("cells (ordered by best match)")
    plt.title(f"overlap block - {len(csel)} cells x {len(nsel)} nuclei  (clean = bright diagonal)")
    plt.colorbar(label="containment"); plt.show()
else:
    print("the block is empty: change BOX / ZHALF, or check that the cell ids match the provenance table")

## 11. Fragment or missed nucleus? Cell size tells them apart

"Cell with no nucleus" is ambiguous. A **tiny** one is probably a fragment of a cell. A **full-sized** one is probably a real cell whose nucleus was dim or deep and got missed. And a "merge" candidate should be **larger** than a clean cell, since it holds two cells' worth. This section compares volumes of each class with the clean cells.

In [ ]:
# Compare the volumes of the anomaly classes with the clean cells.
import matplotlib.pyplot as plt
cp = cell_prov.set_index("dense_id")
vox = cp.voxel_count
clean_ids  = list(nuclei_assigned[nuclei_assigned == 1].index)
nonuc_ids  = list(nuclei_assigned[nuclei_assigned == 0].index)
multi_ids  = list(nuclei_assigned[nuclei_assigned >= 2].index)
clean_med = float(vox.loc[clean_ids].median()) if clean_ids else 0.0
print(f"median clean-cell volume = {clean_med:,.0f} vox  (the 'real cell' reference)")

for name, ids in [("no-nucleus", nonuc_ids), ("multi-nucleus", multi_ids)]:
    if not ids: 
        print(f"{name}: none"); continue
    v = vox.loc[ids]
    frac_small = float((v < 0.3 * clean_med).mean())
    print(f"{name}: n={len(ids):,}  median={v.median():,.0f} vox  "
          f"({100*frac_small:.0f}% are < 30% of a clean cell -> likely fragments)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.logspace(np.log10(max(1, vox.min())), np.log10(vox.max() + 1), 50)
ax[0].hist(vox.loc[clean_ids], bins=bins, alpha=0.6, label="clean 1:1", color="green", density=True)
if nonuc_ids:
    ax[0].hist(vox.loc[nonuc_ids], bins=bins, alpha=0.6, label="no-nucleus", color="orange", density=True)
ax[0].axvline(clean_med, ls="--", c="k", lw=1, label="clean median")
ax[0].set(xscale="log", title="no-nucleus cells: fragment (small) or missed nucleus (clean-sized)",
          xlabel="cell volume (vox, log)", ylabel="density"); ax[0].legend()
ax[1].hist(vox.loc[clean_ids], bins=bins, alpha=0.6, label="clean 1:1", color="green", density=True)
if multi_ids:
    ax[1].hist(vox.loc[multi_ids], bins=bins, alpha=0.6, label="multi-nucleus", color="red", density=True)
ax[1].axvline(clean_med, ls="--", c="k", lw=1)
ax[1].set(xscale="log", title="merge candidates: should be LARGER than clean (2 cells' worth)",
          xlabel="cell volume (vox, log)", ylabel="density"); ax[1].legend()
plt.tight_layout(); plt.show()

## 12. Merge or division? A first guess from the nuclei alone

A cell with two nuclei is either two cells glued together, **or** one cell that is dividing (two daughter nuclei in one cell), or a genuinely two-nucleus cell. Only the cell wall can really decide. But the nuclei give a first guess: dividing cells have two nuclei that are **close** (compared with the cell size) and **similar in size**; glued cells have nuclei about a cell diameter apart, of any size ratio.

The scatter marks the "division-like" corner. It is a prior, not a verdict.

In [ ]:
# Merge vs division prior from the labels alone: divisions have close, similar-sized nuclei;
# merges have nuclei about a cell diameter apart, of any size ratio.
import matplotlib.pyplot as plt

ncp = nuc_prov.assign(z=(nuc_prov.z_min+nuc_prov.z_max)/2, y=(nuc_prov.y_min+nuc_prov.y_max)/2,
                      x=(nuc_prov.x_min+nuc_prov.x_max)/2).set_index('dense_id')
ccp = cell_prov.set_index('dense_id')
cell_diam = ((ccp.voxel_count * 6/np.pi) ** (1/3))   # equivalent-sphere diameter (vox)

# the 2 (or more) nuclei assigned to each merge cell
merge_cells = nuclei_assigned[nuclei_assigned >= 2].index
asg_by_cell = assigned.groupby('cell_id').nucleus_id.apply(list)
rows = []
for cid in merge_cells:
    nids = [n for n in asg_by_cell.get(cid, []) if n in ncp.index]
    if len(nids) < 2:
        continue
    # use the two LARGEST assigned nuclei
    nids = sorted(nids, key=lambda n: -ncp.voxel_count[n])[:2]
    a, b = nids
    d = float(np.sqrt((ncp.z[a]-ncp.z[b])**2 + (ncp.y[a]-ncp.y[b])**2 + (ncp.x[a]-ncp.x[b])**2))
    dn = d / max(float(cell_diam.get(cid, np.nan)), 1e-6)            # normalized by cell size
    vr = min(ncp.voxel_count[a], ncp.voxel_count[b]) / max(ncp.voxel_count[a], ncp.voxel_count[b])
    rows.append((cid, dn, vr))
import pandas as pd
mm = pd.DataFrame(rows, columns=['cell_id', 'dist_norm', 'size_ratio'])
print(f"{len(mm):,} two-nuclei cells profiled")
# division-LIKE prior: nuclei close (<0.5 cell-diam apart) AND similar size (ratio>0.6)
div_like = mm[(mm.dist_norm < 0.5) & (mm.size_ratio > 0.6)]
print(f"  division-LIKE (close + similar-size): {len(div_like):,} ({100*len(div_like)/max(len(mm),1):.0f}%)")
print(f"  merge-LIKE (the rest): {len(mm)-len(div_like):,}")
print("  NOTE: this is only a first guess from the nuclei; the cell wall is what decides. It does tell you roughly how")
print("        many of the two-nucleus cells are dividing cells and how many are real merges.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(mm.dist_norm, mm.size_ratio, s=6, alpha=0.3)
ax[0].axvline(0.5, ls='--', c='r'); ax[0].axhline(0.6, ls='--', c='r')
ax[0].set(xlabel='inter-nucleus dist / cell diameter', ylabel='nuclear size ratio (min/max)',
          title='lower-left box = division-like (close + similar)', xlim=(0, 2))
ax[1].hist(mm.dist_norm.clip(0, 2), bins=40)
ax[1].axvline(0.5, ls='--', c='r')
ax[1].set(xlabel='inter-nucleus dist / cell diameter', ylabel='# cells',
          title='bimodal? close peak = divisions, far peak = merges')
plt.tight_layout(); plt.show()

## 13. A slice coloured by type, with the nuclei outlined

One Z plane of the cell volume, coloured by type (section 6): green for clean 1:1, red for cells holding two or more nuclei, orange for cells with no nucleus. The nuclei in the same plane are outlined in white. Change `Zc` to look at another plane. Nuclei outlines should sit inside the coloured cells; outlines that fall systematically outside or at the edge of cells would point to a shift between the two volumes (section 7 puts a number on that).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from skimage.segmentation import find_boundaries

maxid = int(max(cell_ids)) if cell_ids else 0
lut = np.zeros(maxid + 1, dtype=np.int32)
lut[:] = 0
for cid in cell_ids:
    n = int(nuclei_assigned.get(cid, 0))
    lut[cid] = 1 if n == 1 else (2 if n >= 2 else 3)
Zc = cells_vol.shape[0] // 2   # EDIT: the Z plane to show
sl = np.asarray(cells_vol[Zc]); sl = np.where(sl <= maxid, sl, 0)
cmap = ListedColormap(["black", "#2ca02c", "#d62728", "#ff7f0e"])
rgb = cmap(lut[sl] / 3.0)[..., :3]
outline = find_boundaries(np.asarray(nuclei_vol[Zc]), mode="inner")
rgb[outline] = (1.0, 1.0, 1.0)
plt.figure(figsize=(9, 9)); plt.imshow(rgb, interpolation="nearest")
plt.title(f"Z={Zc}: green=clean 1:1   red=merge (2+ nuclei)   orange=no nucleus   white outline=nuclei")
plt.axis("off"); plt.show()

## 14. The cells to look at first

Two worklists, both from the assignment of section 6: the cells holding two or more nuclei (merge candidates, most nuclei first), and the nuclei that straddle a cell boundary (most evenly split first, measured by the share of the nucleus in its second cell).

In [ ]:
merge_cands = nuclei_assigned[nuclei_assigned >= 2].sort_values(ascending=False)
straddlers = second[second >= TAU_STRUNG].sort_values(ascending=False)
print(f"{len(merge_cands):,} merge candidates (cell with 2 or more nuclei). Top 10, cell_id -> #nuclei:")
print(merge_cands.head(10).to_string())
print(f"\n{len(straddlers):,} straddling nuclei (at least {TAU_STRUNG:.0%} of the nucleus in a second cell).")
print("Top 10, nucleus_id -> share of the nucleus in its second cell:")
print(straddlers.head(10).to_string())

## 15. Every cell and nucleus in exactly one bucket

The whole problem on one screen. The overlap graph (cells and nuclei joined where they overlap) falls into connected groups, and each group has a type:

- **clean**: one cell and one nucleus,
- **merge**: one cell, two or more nuclei,
- **strung_nucleus**: one nucleus across two or more cells,
- **tangled**: several cells and several nuclei in one clump,
- **cell_no_nucleus** and **orphan_nucleus**: a cell or a nucleus with no partner.

The census adds up to the total number of cells and nuclei (the cell checks it), so nothing falls through.

The counts here are by connected group, so they differ from section 6: a cell that holds two nuclei and also shares one of them with a neighbour is one **tangled** group, not a merge plus a straddling nucleus.

In [ ]:
# ===== component census - every cell + every nucleus in exactly one bucket =====
# Build the cell-nucleus overlap graph, split it into connected components, and tally by type.
# Needs the edge table, FLOOR, cell_ids and nucleus_ids from the earlier sections.
import numpy as np
from packages.epicycle_scaledseg.reconcile import build_components, component_census

edge_tuples = list(zip(edges.cell_id.astype(int), edges.nucleus_id.astype(int),
                       edges.containment.astype(float)))
comps = build_components(edge_tuples, cell_ids, nucleus_ids, floor=FLOOR)
cen = component_census(comps)

order = ['clean', 'merge', 'strung_nucleus', 'cell_no_nucleus', 'orphan_nucleus', 'tangled']
note = {'clean': 'nothing to do', 'merge': 'two cells glued, or a dividing cell',
        'strung_nucleus': 'a nucleus across a cell boundary', 'tangled': 'several cells and nuclei in one clump',
        'cell_no_nucleus': 'fragment, or a missed nucleus', 'orphan_nucleus': 'a nucleus outside every cell'}
print(f"{'type':16s} {'components':>11s} {'cells':>9s} {'nuclei':>9s}   what")
tot_c = tot_n = 0
for t in order:
    r = cen.get(t, {'components': 0, 'cells': 0, 'nuclei': 0})
    print(f"{t:16s} {r['components']:>11,} {r['cells']:>9,} {r['nuclei']:>9,}   {note[t]}")
    tot_c += r['cells']; tot_n += r['nuclei']
print(f"{'TOTAL':16s} {len(comps):>11,} {tot_c:>9,} {tot_n:>9,}")
assert tot_c == len(cell_ids) and tot_n == len(nucleus_ids), \
    "every cell + nucleus must be in exactly one component"
print(f"\nsanity: {tot_c:,} cells == {len(cell_ids):,} ids ; {tot_n:,} nuclei == {len(nucleus_ids):,} ids")
print("-> nothing falls through: every cell and every nucleus is in exactly one group.")

## Where next

Notebook 03 uses these two volumes to make them consistent: it cuts straddling nuclei at the cell boundary, rescues missed nuclei, and reports the cells that still hold two nuclei.